Check time alignment between eeg and et data by looking at blinks and artifacts

In [1]:
import os
import sys
import mne
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
sys.path.append('../..')
from data.data_configs import *
from data.data_utils import transform_eeg_time_to_ms, transform_eeg_index_to_ms


### Check difference of max timestamp between EEG and ET
If they are all in a similar range that shows that the time conversion and alignment roughly worked

In [2]:
for subject in ET_SUBJECTS:
    events_df = pd.read_csv(os.path.join(LABELED_ET_DATA_DIR,f"{subject}_labeled_events.csv"),  dtype={'stimulus': str})

    raw_eeg_path = os.path.join(RAW_EEG_DIR, f"{subject}.edf")
    raw = mne.io.read_raw_edf(raw_eeg_path, preload=True, verbose=False)  # preload=True loads data into memory

    eeg_times_ms = transform_eeg_time_to_ms(raw.times)
    et_times_ms = np.asarray(events_df["eeg_time_start"].to_list())

    end_diff = abs(et_times_ms.max() - eeg_times_ms.max())

    print(f"End diff for subject {subject}: {end_diff:.0f} ms")
    break

End diff for subject P01: 9981 ms


## Check time alignement parameter

For every subject I plot the EEG wave of the Frontal channels which best capture blinks. I overlay the blinks recorded by the eyetracker to see if they align with the peaks. This has to be done manually... I tried using blink detection algorithms to check automatically but that was reliable enough ...

Subjects where the et and eeg data cannot be aligned (meaning the b parameter by the authors is wrong) are removed from the data set

In [6]:
def plot_frontal_eeg_with_blinks(raw, et_blink_times_ms, subject_id=None):
    """
    Plot frontal EEG (Fp1/Fp2) with ET blink timestamps as green vertical lines.
    Pauses execution until the Qt plot is closed.
    
    Parameters
    ----------
    raw : mne.io.Raw
        Raw EEG object.
    et_blink_times_ms : array-like
        Eye-tracking blink timestamps in milliseconds.
    subject_id : str or int, optional
        Subject identifier for the plot title.
    """
    # Bandpass filter: remove very low (<0.5 Hz) and very high (>30 Hz) frequencies
    raw_filtered = raw.copy().filter(l_freq=0.5, h_freq=30., fir_design='firwin', verbose=False)
    
    # Convert ET blink times to seconds
    et_blink_times_sec = np.array(et_blink_times_ms) / 1000.0  # ms -> s
    
    # Convert ET blink times to EEG sample indices
    sfreq = raw_filtered.info['sfreq']
    et_blink_samples = (et_blink_times_sec * sfreq).astype(int)
    
    # Create MNE events array
    events = np.zeros((len(et_blink_samples), 3), dtype=int)
    events[:, 0] = et_blink_samples
    events[:, 2] = 1  # blink event ID
    
    # Create annotations for visualization
    annotations = mne.Annotations(
        onset=et_blink_times_sec,
        duration=np.zeros_like(et_blink_times_sec),
        description=['blink']*len(et_blink_times_sec)
    )
    raw_filtered.set_annotations(annotations)
    
    # Print sanity check
    print(f"Subject {subject_id} EEG start/end (s): {raw_filtered.times[0]:.2f} / {raw_filtered.times[-1]:.2f}")
    print(f"Subject {subject_id} ET blink start/end (s): {et_blink_times_sec.min():.2f} / {et_blink_times_sec.max():.2f}")
    
    # Pick frontal channels
    frontal_raw = raw_filtered.copy().pick(DEFAULT_EEGNET_CHANNELS)
    
    # Enable Qt backend for scrollable plot
    %matplotlib qt
    
    # Plot and block until closed
    title = f"Subject {subject_id} - Frontal EEG with ET blinks" if subject_id else "Frontal EEG with ET blinks"
    frontal_raw.plot(
        scalings={'eeg': 100e-6},
        show=True,
        block=True,                # block execution until window is closed
        title=title,
        events=events,
        event_color={1: 'green'},
        verbose=False
    )


In [11]:
for subject in ET_SUBJECTS[15:]:
    events_df = pd.read_csv(os.path.join(LABELED_ET_DATA_DIR,f"{subject}_labeled_events.csv"),  dtype={'stimulus': str})

    blinks_df = events_df[events_df["event_type"] == "Blink"]
    blinks_df = blinks_df.sort_values(by='start_time')
    et_blinks = blinks_df["eeg_time_start"].to_list()

    raw_eeg_path = os.path.join(RAW_EEG_DIR, f"{subject}.edf")
    raw_eeg = mne.io.read_raw_edf(raw_eeg_path, preload=True, verbose=False)

    plot_frontal_eeg_with_blinks(raw_eeg, et_blinks, subject_id=subject)


Subject P20 EEG start/end (s): 0.00 / 2440.00
Subject P20 ET blink start/end (s): 322.35 / 2433.50
Subject P21 EEG start/end (s): 0.00 / 2107.00
Subject P21 ET blink start/end (s): 281.27 / 2099.83
Subject P23 EEG start/end (s): 0.00 / 2448.00
Subject P23 ET blink start/end (s): 517.30 / 2452.10
Subject P24 EEG start/end (s): 0.00 / 2385.00
Subject P24 ET blink start/end (s): 265.25 / 2381.29
Subject P25 EEG start/end (s): 0.00 / 2492.00
Subject P25 ET blink start/end (s): 443.70 / 2496.03
Subject P26 EEG start/end (s): 0.00 / 2311.00
Subject P26 ET blink start/end (s): 223.69 / 2313.32
Subject P27 EEG start/end (s): 0.00 / 2128.00
Subject P27 ET blink start/end (s): 279.13 / 2120.12
Subject P28 EEG start/end (s): 0.00 / 2259.00
Subject P28 ET blink start/end (s): 389.63 / 2255.36
Subject P29 EEG start/end (s): 0.00 / 2114.00
Subject P29 ET blink start/end (s): 636.26 / 2479.56
Subject P30 EEG start/end (s): 0.00 / 2426.00
Subject P30 ET blink start/end (s): 429.18 / 2419.52
Subject P3